# Data Quality & Governance Framework — Walkthrough

This notebook walks through the framework interactively: generate messy data, run the DQ engine, inspect scores column by column, and visualize the health scorecard.

Run `pip install -r ../requirements.txt` first if needed.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
from generate_sample_data import generate_raw_dataset
from data_quality_checks import DataQualityChecker, DEFAULT_LOAN_VALIDITY_RULES

df = generate_raw_dataset()
print(df.shape)
df.head()

## Run the Data Quality checks

In [ ]:
checker = DataQualityChecker(
    df=df,
    key_column='customer_id',
    validity_rules=DEFAULT_LOAN_VALIDITY_RULES,
).run()

checker.column_scores_

In [ ]:
checker.summary()

## Visualize the scorecard

In [ ]:
import matplotlib.pyplot as plt

scores = checker.column_scores_.sort_values('health_score')
colors = scores['status'].map({'Good': 'green', 'Warning': 'orange', 'Critical': 'red'})

plt.figure(figsize=(8, 5))
plt.barh(scores['column'], scores['health_score'], color=colors)
plt.xlabel('Health Score')
plt.title('Data Quality Health Scorecard')
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

## Drill into the row-level issues log

In [ ]:
checker.issues_log_['issue_type'].value_counts()

## Next: run the full pipeline (SQL + reports) from the command line

```bash
cd ..
python run_pipeline.py
```
This writes `reports/column_health_scorecard.csv`, `reports/overall_summary.json`, and `reports/issues_log.csv`, and builds the cleaned table via SQLite (see `src/sql_pipeline.py`).